## Hyperparameter Tuning for GRU

In [ ]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
    !pip install -q optuna
else:
    PROJECT_ROOT = Path('..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 32.1 MB/s eta 0:00:00
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [ ]:
import json

import numpy as np
import optuna

from src.dataloader import load_cgm_stats, load_tensors, load_participant_ids
from src import metrics
from src import train
from src.models.nn_baseline import GRUPredictor, HORIZONS

optuna.logging.set_verbosity(optuna.logging.WARNING)

MODEL_NAME = 'GRU-tuned'

TRIAL_SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
N_TRIALS = 40

SEARCH_SEED = 0  # seeds Optuna's RandomSampler itself

FINAL_SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
DEFAULT_PARAMS = dict(hidden_size=128, num_layers=2, dropout=0.2, lr=1e-3, weight_decay=1e-4,
                       batch_size=256)

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '2_best_cgm_encoder'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'best_gru'
TRIALS_CSV_PATH = RESULTS_DIR / 'gru_tuning_trials.csv'

FINAL_SUMMARY_PATH = RESULTS_DIR / 'gru_tuning_final_summary.json'
VAL_FINAL_SUMMARY_PATH = RESULTS_DIR / 'gru_tuning_val_final_summary.json'

DEFAULT_VAL_SUMMARY_PATH = PROJECT_ROOT / 'experiments' / '1_baseline' / 'results' / 'gru' / 'gru_val_stability_summary.json'
DEFAULT_SUMMARY_PATH = PROJECT_ROOT / 'experiments' / '1_baseline' / 'results' / 'gru' / 'gru_stability_summary.json'

In [ ]:
stats = load_cgm_stats()
val_loader = load_tensors('val', batch_size=256, shuffle=False)
test_loader = load_tensors('test', batch_size=256, shuffle=False)

val_participant_ids = load_participant_ids('val')
test_participant_ids = load_participant_ids('test')
VAL_PARTICIPANT_IDS = sorted(set(val_participant_ids))
PARTICIPANT_IDS = sorted(set(test_participant_ids))


print(f'number of validation batches: ', len(val_loader),
      '\nnumber of test batches: ', len(test_loader))
print()
print('number of validation participants:', len(VAL_PARTICIPANT_IDS))
print('Corresponding participants ID: \n', VAL_PARTICIPANT_IDS)
print()
print('number of test participants:', len(PARTICIPANT_IDS))
print('Corresponding participants ID: \n', PARTICIPANT_IDS)


number of validation batches:  66 
number of test batches:  66

number of validation participants: 6
Corresponding participants ID: 
 [np.str_('CGMacros-005'), np.str_('CGMacros-016'), np.str_('CGMacros-027'), np.str_('CGMacros-044'), np.str_('CGMacros-046'), np.str_('CGMacros-048')]

number of test participants: 6
Corresponding participants ID: 
 [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]


In [ ]:
def objective(trial):
    hidden_size = trial.suggest_categorical('hidden_size', [64, 128, 256])
    num_layers = trial.suggest_categorical('num_layers', [1, 2, 3])
    dropout = trial.suggest_categorical('dropout', [0.1, 0.2, 0.3])
    lr = trial.suggest_categorical('lr', [1e-3, 1e-4, 1e-5])
    weight_decay = trial.suggest_categorical('weight_decay', [1e-4, 1e-5, 1e-6])
    batch_size = trial.suggest_categorical('batch_size', [64, 128, 256, 512])

    print(f"\n{'-' * 16}trial={trial.number}{'-' * 16}")
    print(f"hidden_size={hidden_size}  num_layers={num_layers}  dropout={dropout:.3f}  "
          f"lr={lr:.2e}  weight_decay={weight_decay:.2e}  batch_size={batch_size}")

    trial_train_loader = load_tensors('train', batch_size=batch_size)

    seed_val_losses = []
    for s in TRIAL_SEEDS:
        train.set_seed(s)
        model = GRUPredictor(hidden_size=hidden_size, num_layers=num_layers, dropout=dropout)
        model, history = train.train_model(
            model, trial_train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
            lr=lr, weight_decay=weight_decay,
            checkpoint_dir=EXPERIMENT_DIR / 'checkpoints' / f'trial{trial.number}_seed{s}',
            verbose=False, progress_bar=True, progress_desc=f'trial={trial.number} seed={s}',
        )
        best_val_loss = min(history['val_loss'])
        seed_val_losses.append(best_val_loss)
        print(f"  trial={trial.number}  seed={s}  best_val_loss={best_val_loss:.4f}")

    mean_val_loss = float(np.mean(seed_val_losses))
    trial.set_user_attr('seed_val_losses', seed_val_losses)
    print(f"trial={trial.number:>3}  mean_val_loss over {len(TRIAL_SEEDS)} seeds = "
          f"{mean_val_loss:.4f}")
    return mean_val_loss

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sampler = optuna.samplers.RandomSampler(seed=SEARCH_SEED)
study = optuna.create_study(direction='minimize', sampler=sampler)
study.enqueue_trial(DEFAULT_PARAMS)
study.optimize(objective, n_trials=N_TRIALS)

print(f"\n=== best trial: #{study.best_trial.number} ===")
print('best_params:', study.best_params)
print(f'best mean_val_loss (over {len(TRIAL_SEEDS)} seeds):', study.best_value)

trials_df = study.trials_dataframe()
trials_df.to_csv(TRIALS_CSV_PATH, index=False)
print(f'\nsaved all {N_TRIALS} trials to {TRIALS_CSV_PATH}')

completed = sorted((t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE),
                    key=lambda t: t.value)
default_rank = next(i for i, t in enumerate(completed, start=1) if t.params == DEFAULT_PARAMS)
print(f'default config ranked #{default_rank} of {len(completed)} by mean_val_loss')


----------------trial=0----------------
hidden_size=128  num_layers=2  dropout=0.200  lr=1.00e-03  weight_decay=1.00e-04  batch_size=256
device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: GRUPredictor  params: 157,957


trial=0 seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

  trial=0  seed=7  best_val_loss=0.3260
device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: GRUPredictor  params: 157,957


trial=0 seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

  trial=0  seed=14  best_val_loss=0.3250
device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: GRUPredictor  params: 157,957


...

  trial=39  seed=49  best_val_loss=0.3256
device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: GRUPredictor  params: 257,029


trial=39 seed=56:   0%|          | 0/50 [00:00<?, ?it/s]

  trial=39  seed=56  best_val_loss=0.3339
device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: GRUPredictor  params: 257,029


trial=39 seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

  trial=39  seed=63  best_val_loss=0.3265
trial= 39  mean_val_loss over 9 seeds = 0.3301

=== best trial: #5 ===
best_params: {'hidden_size': 256, 'num_layers': 3, 'dropout': 0.2, 'lr': 0.0001, 'weight_decay': 1e-05, 'batch_size': 256}
best mean_val_loss (over 9 seeds): 0.32605399356948006

saved all 40 trials to /content/drive/MyDrive/dissertation/experiments/2_best_cgm_encoder/results/best_gru/gru_tuning_trials.csv
default config ranked #4 of 40 by mean_val_loss


## Final confirmatory evaluation

In [ ]:
best_params = study.best_params
best_train_loader = load_tensors('train', batch_size=best_params['batch_size'])

winner_seed_results = []
for seed in FINAL_SEEDS:
    print(f"Validation results — {MODEL_NAME} | seed={seed}\n")

    train.set_seed(seed)
    model = GRUPredictor(hidden_size=best_params['hidden_size'],
                          num_layers=best_params['num_layers'],
                          dropout=best_params['dropout'])

    model, history = train.train_model(
        model, best_train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
        lr=best_params['lr'], weight_decay=best_params['weight_decay'],
        checkpoint_dir=EXPERIMENT_DIR / 'checkpoints' / f'winner_seed{seed}',
        verbose=False, progress_bar=True, progress_desc=f'winner/seed={seed}',
    )

    y_pred_val, y_true_val = train.predict(model, val_loader)
    y_pred_test, y_true_test = train.predict(model, test_loader)
    val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
    test_results = metrics.evaluate(y_pred_test, y_true_test, stats['cgm_mean'], stats['cgm_std'])

    val_results_per_participant = metrics.evaluate_per_participant(
        y_pred_val, y_true_val, val_participant_ids, stats['cgm_mean'], stats['cgm_std'])
    test_results_per_participant = metrics.evaluate_per_participant(
        y_pred_test, y_true_test, test_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    winner_seed_results.append({
        'seed': seed, 'val_results': val_results, 'test_results': test_results,
        'val_results_per_participant': val_results_per_participant,
        'test_results_per_participant': test_results_per_participant,
    })

    val_rmse = [round(val_results[h]['rmse'], 2) for h in HORIZONS]
    print(f"seed={seed:>3}  val RMSE @ {HORIZONS} min = {val_rmse}")

Validation results — GRU-tuned | seed=7

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: GRUPredictor  params: 1,021,957


winner/seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

seed=  7  val RMSE @ [15, 30, 60, 90, 120] min = [11.02, 17.5, 24.8, 29.02, 31.62]
Validation results — GRU-tuned | seed=14

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: GRUPredictor  params: 1,021,957


winner/seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

seed= 14  val RMSE @ [15, 30, 60, 90, 120] min = [10.85, 17.4, 24.86, 29.17, 31.83]
Validation results — GRU-tuned | seed=21

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: GRUPredictor  params: 1,021,957


...

seed= 56  val RMSE @ [15, 30, 60, 90, 120] min = [10.79, 17.34, 24.84, 29.09, 31.65]
Validation results — GRU-tuned | seed=63

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: GRUPredictor  params: 1,021,957


winner/seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

seed= 63  val RMSE @ [15, 30, 60, 90, 120] min = [11.08, 17.56, 24.9, 29.11, 31.71]
Validation results — GRU-tuned | seed=70

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: GRUPredictor  params: 1,021,957


winner/seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

seed= 70  val RMSE @ [15, 30, 60, 90, 120] min = [10.81, 17.39, 24.84, 29.08, 31.64]


In [ ]:
def _extract(participant_result, metric):
    if metric == 'zone_a':
        return participant_result['ceg']['A']
    if metric == 'zone_ab':
        return participant_result['ceg']['A'] + participant_result['ceg']['B']
    return participant_result[metric]

METRICS = ['rmse', 'mae', 'grmse', 'zone_a', 'zone_ab']

def build_summary(results_key, participant_key, participant_ids):
    """results_key/participant_key: ('val_results', 'val_results_per_participant') or
    ('test_results', 'test_results_per_participant')."""
    summary = {}
    for h in HORIZONS:
        summary[h] = {}
        for m in METRICS:
            if m == 'zone_a':
                seed_values = [r[results_key][h]['ceg']['A'] for r in winner_seed_results]
            elif m == 'zone_ab':
                seed_values = [r[results_key][h]['ceg']['A'] + r[results_key][h]['ceg']['B']
                                for r in winner_seed_results]
            else:
                seed_values = [r[results_key][h][m] for r in winner_seed_results]

            mean = float(np.mean(seed_values))
            sd1 = float(np.std(seed_values, ddof=1))

            participant_means = []
            for pid in participant_ids:
                values_across_seeds = [_extract(r[participant_key][pid][h], m)
                                        for r in winner_seed_results]
                participant_means.append(np.mean(values_across_seeds))
            sd2 = float(np.std(participant_means, ddof=1))

            summary[h][m] = {'mean': mean, 'sd1_across_seeds': sd1, 'sd2_across_participants': sd2}
    return summary

In [ ]:
val_summary = build_summary('val_results', 'val_results_per_participant', VAL_PARTICIPANT_IDS)
test_summary = build_summary('test_results', 'test_results_per_participant', PARTICIPANT_IDS)

In [ ]:
def _print_summary(split_label, summary):
    print(f'=== {MODEL_NAME} {split_label} set: overall mean +/- SD_seed (SD_participant) ===')
    for h in HORIZONS:
        s = summary[h]
        print(
            f"{h:>3}-min:  "
            f"RMSE = {s['rmse']['mean']:.2f} +/- {s['rmse']['sd1_across_seeds']:.2f} "
            f"({s['rmse']['sd2_across_participants']:.2f}) mg/dL   "
            f"MAE = {s['mae']['mean']:.2f} +/- {s['mae']['sd1_across_seeds']:.2f} "
            f"({s['mae']['sd2_across_participants']:.2f})   "
            f"gRMSE = {s['grmse']['mean']:.2f} +/- {s['grmse']['sd1_across_seeds']:.2f} "
            f"({s['grmse']['sd2_across_participants']:.2f})   "
            f"Zone A = {s['zone_a']['mean']:.2f} +/- {s['zone_a']['sd1_across_seeds']:.2f} "
            f"({s['zone_a']['sd2_across_participants']:.2f})%   "
            f"Zone A+B = {s['zone_ab']['mean']:.2f} +/- {s['zone_ab']['sd1_across_seeds']:.2f} "
            f"({s['zone_ab']['sd2_across_participants']:.2f})%"
        )

_print_summary('validation', val_summary)
print()
_print_summary('test', test_summary)

with open(VAL_FINAL_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'best_params': best_params,
        'search_seed': SEARCH_SEED,
        'trial_seeds': TRIAL_SEEDS,
        'n_trials': N_TRIALS,
        'final_seeds': FINAL_SEEDS,
        'per_horizon_summary': val_summary,
    }, f, indent=2)

with open(FINAL_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'best_params': best_params,
        'search_seed': SEARCH_SEED,
        'trial_seeds': TRIAL_SEEDS,
        'n_trials': N_TRIALS,
        'final_seeds': FINAL_SEEDS,
        'per_horizon_summary': test_summary,
    }, f, indent=2)

print(f'\nsaved validation (selection) summary to {VAL_FINAL_SUMMARY_PATH}')
print(f'saved test (confirmatory-only) summary to {FINAL_SUMMARY_PATH}')

=== GRU-tuned validation set: overall mean +/- SD_seed (SD_participant) ===
 15-min:  RMSE = 10.91 +/- 0.09 (2.34) mg/dL   MAE = 7.35 +/- 0.12 (1.55)   gRMSE = 11.83 +/- 0.15 (3.18)   Zone A = 96.76 +/- 0.16 (3.10)%   Zone A+B = 99.66 +/- 0.02 (0.67)%
 30-min:  RMSE = 17.43 +/- 0.06 (3.79) mg/dL   MAE = 11.60 +/- 0.15 (2.41)   gRMSE = 19.60 +/- 0.13 (5.64)   Zone A = 89.92 +/- 0.20 (5.58)%   Zone A+B = 99.34 +/- 0.02 (0.97)%
 60-min:  RMSE = 24.84 +/- 0.03 (7.40) mg/dL   MAE = 16.52 +/- 0.17 (4.70)   gRMSE = 29.36 +/- 0.10 (11.05)   Zone A = 81.92 +/- 0.24 (5.25)%   Zone A+B = 98.67 +/- 0.02 (1.45)%
 90-min:  RMSE = 29.10 +/- 0.04 (10.34) mg/dL   MAE = 19.59 +/- 0.20 (6.80)   gRMSE = 35.08 +/- 0.09 (15.16)   Zone A = 77.03 +/- 0.41 (7.17)%   Zone A+B = 98.21 +/- 0.03 (1.96)%
120-min:  RMSE = 31.71 +/- 0.06 (12.13) mg/dL   MAE = 21.71 +/- 0.18 (8.27)   gRMSE = 38.57 +/- 0.10 (17.86)   Zone A = 73.21 +/- 0.42 (8.49)%   Zone A+B = 97.89 +/- 0.03 (2.43)%

=== GRU-tuned test set: overall me

In [ ]:
with open(DEFAULT_VAL_SUMMARY_PATH) as f:
    default_val_summary = json.load(f)['per_horizon_summary']

print(f"{'Horizon':>9} | {'Default val RMSE':>22} | {'Tuned val RMSE':>22} | {'Diff':>7}")
print('-' * 72)
for h in HORIZONS:
    d = default_val_summary[str(h)]['rmse']
    t = val_summary[h]['rmse']
    diff = t['mean'] - d['mean']
    print(f"{h:>6} min | "
          f"{d['mean']:6.2f} +/- {d['sd1_across_seeds']:.2f} ({d['sd2_across_participants']:5.2f}) | "
          f"{t['mean']:6.2f} +/- {t['sd1_across_seeds']:.2f} ({t['sd2_across_participants']:5.2f}) | "
          f"{diff:+.2f}")

  Horizon |       Default val RMSE |         Tuned val RMSE |    Diff
------------------------------------------------------------------------
    15 min |  10.96 +/- 0.09 ( 2.38) |  10.91 +/- 0.09 ( 2.34) | -0.06
    30 min |  17.48 +/- 0.10 ( 3.80) |  17.43 +/- 0.06 ( 3.79) | -0.05
    60 min |  24.89 +/- 0.08 ( 7.35) |  24.84 +/- 0.03 ( 7.40) | -0.05
    90 min |  29.16 +/- 0.10 (10.25) |  29.10 +/- 0.04 (10.34) | -0.07
   120 min |  31.76 +/- 0.11 (12.02) |  31.71 +/- 0.06 (12.13) | -0.05


In [ ]:
with open(DEFAULT_SUMMARY_PATH) as f:
    default_summary = json.load(f)['per_horizon_summary']

print(f"{'Horizon':>9} | {'Default test RMSE':>22} | {'Tuned test RMSE':>22} | {'Diff':>7}")
print('-' * 72)
for h in HORIZONS:
    d = default_summary[str(h)]['rmse']
    t = test_summary[h]['rmse']
    diff = t['mean'] - d['mean']
    print(f"{h:>6} min | "
          f"{d['mean']:6.2f} +/- {d['sd1_across_seeds']:.2f} ({d['sd2_across_participants']:5.2f}) | "
          f"{t['mean']:6.2f} +/- {t['sd1_across_seeds']:.2f} ({t['sd2_across_participants']:5.2f}) | "
          f"{diff:+.2f}")

  Horizon |      Default test RMSE |        Tuned test RMSE |    Diff
------------------------------------------------------------------------
    15 min |  11.36 +/- 0.18 ( 2.07) |  11.26 +/- 0.12 ( 1.89) | -0.10
    30 min |  18.66 +/- 0.14 ( 3.97) |  18.58 +/- 0.11 ( 3.88) | -0.08
    60 min |  27.59 +/- 0.15 ( 9.18) |  27.61 +/- 0.18 ( 9.32) | +0.02
    90 min |  33.02 +/- 0.24 (12.99) |  33.10 +/- 0.23 (13.32) | +0.08
   120 min |  36.23 +/- 0.30 (15.51) |  36.12 +/- 0.24 (15.83) | -0.11
